In [ ]:
!pip install uv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.1/27.1 MB 34.4 MB/s eta 0:00:00


In [ ]:
!uv pip install -r https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/refs/heads/main/requirements.txt --system

Using Python 3.12.13 environment at: /usr
Resolved 133 packages in 974ms
Prepared 8 packages in 2.42s
Uninstalled 1 package in 26ms
Installed 8 packages in 659ms
 + async-lru==2.3.0
 + jedi==0.20.0
 + json5==0.15.0
 + jupyter-builder==1.1.0
 + jupyter-lsp==2.3.1
 - jupyter-server==2.18.2
 + jupyter-server==2.20.0
 + jupyterlab==4.6.1
 + jupyterlab-server==2.28.0


In [ ]:
import os
import urllib.request

if not os.path.exists("the-verdict.txt"):
  url=("https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/refs/heads/main/ch02/01_main-chapter-code/the-verdict.txt")
  file_path="the-verdict.txt"
  urllib.request.urlretrieve(url,file_path)

In [ ]:
with open("the-verdict.txt","r",encoding="utf-8") as f:
  raw_text=f.read()

In [ ]:
raw_text

'I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that, in the height of his glory, he had dropped his painting, married a rich widow, and established himself in a villa on the Riviera. (Though I rather thought it would have been Rome or Florence.)\n\n"The height of his glory"--that was what the women called it. I can hear Mrs. Gideon Thwing--his last Chicago sitter--deploring his unaccountable abdication. "Of course it\'s going to send the value of my picture \'way up; but I don\'t think of that, Mr. Rickham--the loss to Arrt is all I think of." The word, on Mrs. Thwing\'s lips, multiplied its _rs_ as though they were reflected in an endless vista of mirrors. And it was not only the Mrs. Thwings who mourned. Had not the exquisite Hermia Croft, at the last Grafton Gallery show, stopped me before Gisburn\'s "Moon-dancers" to say, with tears in her eyes: "We shall not look upon its like again"?\n\nWell!--even 

In [ ]:
len(raw_text)

20479

In [ ]:
import re

text="Hello everyone! My name is --Ishita."
result=re.split(r'([,.:;"?!_()]|--|\s)',raw_text)

In [ ]:
result=[item.strip() for item in result if item.strip()]
preprocessed=result

In [ ]:
len(preprocessed)

4519

In [ ]:
preprocessed[:10]

['I',
 'HAD',
 'always',
 'thought',
 'Jack',
 'Gisburn',
 'rather',
 'a',
 'cheap',
 'genius']

In [ ]:
all_words=sorted(set(preprocessed))
all_words

['!',
 '"',
 "'",
 "'Are",
 "'It's",
 "'coming'",
 "'done'",
 "'subject",
 "'technique'",
 "'way",
 '(',
 ')',
 ',',
 '--',
 '.',
 ':',
 ';',
 '?',
 'A',
 'Ah',
 'Among',
 'And',
 'Arrt',
 'As',
 'At',
 'Be',
 'Begin',
 'Burlington',
 'But',
 'By',
 'Carlo',
 'Chicago',
 'Claude',
 'Come',
 'Croft',
 'Destroyed',
 'Devonshire',
 "Don't",
 'Dubarry',
 'Emperors',
 'Florence',
 'For',
 'Gallery',
 'Gideon',
 'Gisburn',
 "Gisburn's",
 'Gisburns',
 'Grafton',
 'Greek',
 'Grindle',
 "Grindle's",
 'Grindles',
 'HAD',
 'Had',
 'Hang',
 'Has',
 'He',
 'Her',
 'Hermia',
 "Hermia's",
 'His',
 'How',
 'I',
 "I'd",
 "I'll",
 "I've",
 'If',
 'In',
 'It',
 "It's",
 'Jack',
 "Jack's",
 'Jove',
 'Just',
 'Lord',
 'Made',
 'Miss',
 "Money's",
 'Monte',
 'Moon-dancers',
 'Mr',
 'Mrs',
 'My',
 'Never',
 'No',
 'Now',
 'Nutley',
 'Of',
 'Oh',
 'On',
 'Once',
 'Only',
 'Or',
 'Perhaps',
 'Poor',
 'Professional',
 'Renaissance',
 'Rickham',
 'Riviera',
 'Rome',
 'Russian',
 'Sevres',
 'She',
 "She's",
 'Str

In [ ]:
vocab_size=len(all_words)

In [ ]:
vocab={token:integer for integer,token in enumerate(all_words)}

In [ ]:
class SimpleTokenizerV1:
    def __init__(self,vocab):
      self.str_to_int = vocab
      self.int_to_str ={i:s for s,i in vocab.items()}

    def encode(self,text):
      # Use the same regex as used for creating the vocabulary
      preprocessed=re.split(r'([,.:;"?!_()]|--|\s)',text)
      preprocessed=[item.strip() for item in preprocessed if item.strip()]
      ids=[self.str_to_int[s] for s in preprocessed]
      return ids

    def decode(self,ids):
      text=" ".join([self.int_to_str[i] for i in ids])
      #Replace spaces before punctuations
      text=re.sub(r'\s+([,.?!"()\'])',r'\1',text)
      return text

In [ ]:
tokenizer = SimpleTokenizerV1(vocab)

In [ ]:
text=""""It's the last he painted, you know,"
      Mrs. Gisburn said with pardonable pride."""

In [ ]:
ids = tokenizer.encode(text)
tokenizer.decode(ids)

'" It\'s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

# Adding Special Context Tokens

In [ ]:
text="Hello! Do you like tea?"

In [ ]:
all_tokens=sorted(list(set(preprocessed)))
all_tokens.extend(["<|endOftext|>","<|unk|>"])

vocab={token:integer for integer,token in enumerate(all_tokens)}

In [ ]:
class SimpleTokenizerV2:
    def __init__(self,vocab):
      self.str_to_int = vocab
      self.int_to_str ={i:s for s,i in vocab.items()}

    def encode(self,text):
      # Use the same regex as used for creating the vocabulary
      preprocessed=re.split(r'([,.:;"?!_()]|--|\s)',text)
      preprocessed=[item.strip() for item in preprocessed if item.strip()]
      preprocessed=[item if item in self.str_to_int else "<|unk|>" for item in preprocessed]
      ids=[self.str_to_int[s] for s in preprocessed]
      return ids

    def decode(self,ids):
      text=" ".join([self.int_to_str[i] for i in ids])
      #Replace spaces before punctuations
      text=re.sub(r'\s+([,.?!"()\'])',r'\1',text)
      return text

In [ ]:
vocab

{'!': 0,
 '"': 1,
 "'": 2,
 "'Are": 3,
 "'It's": 4,
 "'coming'": 5,
 "'done'": 6,
 "'subject": 7,
 "'technique'": 8,
 "'way": 9,
 '(': 10,
 ')': 11,
 ',': 12,
 '--': 13,
 '.': 14,
 ':': 15,
 ';': 16,
 '?': 17,
 'A': 18,
 'Ah': 19,
 'Among': 20,
 'And': 21,
 'Arrt': 22,
 'As': 23,
 'At': 24,
 'Be': 25,
 'Begin': 26,
 'Burlington': 27,
 'But': 28,
 'By': 29,
 'Carlo': 30,
 'Chicago': 31,
 'Claude': 32,
 'Come': 33,
 'Croft': 34,
 'Destroyed': 35,
 'Devonshire': 36,
 "Don't": 37,
 'Dubarry': 38,
 'Emperors': 39,
 'Florence': 40,
 'For': 41,
 'Gallery': 42,
 'Gideon': 43,
 'Gisburn': 44,
 "Gisburn's": 45,
 'Gisburns': 46,
 'Grafton': 47,
 'Greek': 48,
 'Grindle': 49,
 "Grindle's": 50,
 'Grindles': 51,
 'HAD': 52,
 'Had': 53,
 'Hang': 54,
 'Has': 55,
 'He': 56,
 'Her': 57,
 'Hermia': 58,
 "Hermia's": 59,
 'His': 60,
 'How': 61,
 'I': 62,
 "I'd": 63,
 "I'll": 64,
 "I've": 65,
 'If': 66,
 'In': 67,
 'It': 68,
 "It's": 69,
 'Jack': 70,
 "Jack's": 71,
 'Jove': 72,
 'Just': 73,
 'Lord': 74,
 'Ma

In [ ]:
for i, item in enumerate(list(vocab.items())[-5:]):
  print(item)

('younger', 1151)
('your', 1152)
('yourself', 1153)
('<|endOftext|>', 1154)
('<|unk|>', 1155)


In [ ]:
tokenizer=SimpleTokenizerV2(vocab)

In [ ]:
tokenizer.decode(tokenizer.encode(text))

'<|unk|>! <|unk|> you like tea?'

# Byte Pair Encoding

In [ ]:
import tiktoken

In [ ]:
tiktoken.__version__

'0.13.0'

In [ ]:
tokenizer=tiktoken.get_encoding("gpt2")

In [ ]:
text=(
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces jkfhkhf helloplace"
    "of some unknown place"
    )
tokenizer.encode(text,allowed_special={"<|endoftext|>"})

[15496,
 11,
 466,
 345,
 588,
 8887,
 30,
 220,
 50256,
 554,
 262,
 4252,
 18250,
 8812,
 2114,
 474,
 74,
 69,
 71,
 14636,
 69,
 5968,
 20106,
 558,
 1659,
 617,
 6439,
 1295]

## Data Sampling with a Sliding Window

In [ ]:
with open("the-verdict.txt","r",encoding="utf-8") as ob:
  raw_text=ob.read()

enc_text=tokenizer.encode(raw_text)
print(len(enc_text))

5145


In [ ]:
enc_sample=enc_text[50:]

In [ ]:
context_size=4

In [ ]:
x=enc_sample[:context_size]
y=enc_sample[1:context_size+1]

print(f"x: {x}")
print(f"y:  {y}")

x: [290, 4920, 2241, 287]
y:  [4920, 2241, 287, 257]


In [ ]:
import torch

In [ ]:
torch.__version__

'2.11.0+cpu'

In [ ]:
from torch.utils.data import Dataset,DataLoader
class GPTDatasetV1(Dataset):
  def __init__(self,txt,tokenizer,max_length,stride):
    self.input_ids=[]
    self.target_ids=[]
    #Tokenize the entire text
    token_ids=tokenizer.encode(txt,allowed_special={"<|endoftext|>"})

    #Use a sliding window to chunk the book into overlapping sequences of max_length
    for i in range(0,len(token_ids)-max_length,stride):
      input_chunk=token_ids[i:i+max_length]
      target_chunk=token_ids[i+1:i+max_length+1]
      self.input_ids.append(torch.tensor(input_chunk))
      self.target_ids.append(torch.tensor(target_chunk))

  def __len__(self):
    return len(self.input_ids)

  def __getitem__(self,idx):
    return self.input_ids[idx],self.target_ids[idx]

In [ ]:
def create_dataloader_v1(txt,batch_size=2,max_length=256,stride=128,shuffle=True,drop_last=True,num_workers=0):
  #Initialize the tokenizer
  tokenizer=tiktoken.get_encoding("gpt2")

  #Create dataset
  dataset=GPTDatasetV1(txt,tokenizer,max_length,stride)

  #Create dataloader
  dataloader=DataLoader(
      dataset,
      batch_size=batch_size,
      shuffle=shuffle,
      drop_last=drop_last,
      num_workers=num_workers
  )
  return dataloader

In [ ]:
with open("the-verdict.txt","r",encoding="utf-8") as f:
  raw_text=f.read()

In [ ]:
dataloader=create_dataloader_v1(
    raw_text,batch_size=1,max_length=4,stride=1,shuffle=False
)

data_iter=iter(dataloader)
first_batch=next(data_iter)

print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [ ]:
second_batch=next(data_iter)
print(second_batch)


[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


In [ ]:
dataloader=create_dataloader_v1(
    raw_text,batch_size=8,max_length=4,stride=4,shuffle=False
)

data_iter=iter(dataloader)
inputs,targets=next(data_iter)

print("Inputs:\n",inputs)
print("\nTargets:\n",targets)

Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


## Creating Token Embeddings

In [ ]:
input_ids=torch.tensor([2,3,5,1])

In [ ]:
vocab_size=6
output_dim=3

torch.manual_seed(123)
embedding_layer=torch.nn.Embedding(vocab_size,output_dim)

In [ ]:
print(embedding_layer.weight)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


In [ ]:
embedding_layer(torch.tensor([3]))

tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)

In [ ]:
embedding_layer(input_ids)

tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)

### Encoding word positions

In [ ]:
vocab_size=50257
output_dim=256

token_embedding_layer=torch.nn.Embedding(vocab_size,output_dim)

In [ ]:
max_length=4
dataloader=create_dataloader_v1(
    raw_text,batch_size=8,max_length=max_length,stride=max_length,shuffle=False
)
data_iter=iter(dataloader)
inputs,targets=next(data_iter)

In [ ]:
print("Token ids\n:",inputs)
print("\nInputs Shape:\n",inputs.shape)

Token ids
: tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Inputs Shape:
 torch.Size([8, 4])


In [ ]:
token_embeddings=token_embedding_layer(inputs)
print(token_embeddings.shape)
print(token_embeddings)

torch.Size([8, 4, 256])
tensor([[[ 0.4913,  1.1239,  1.4588,  ..., -0.3995, -1.8735, -0.1445],
         [ 0.4481,  0.2536, -0.2655,  ...,  0.4997, -1.1991, -1.1844],
         [-0.2507, -0.0546,  0.6687,  ...,  0.9618,  2.3737, -0.0528],
         [ 0.9457,  0.8657,  1.6191,  ..., -0.4544, -0.7460,  0.3483]],

        [[ 1.5460,  1.7368, -0.7848,  ..., -0.1004,  0.8584, -0.3421],
         [-1.8622, -0.1914, -0.3812,  ...,  1.1220, -0.3496,  0.6091],
         [ 1.9847, -0.6483, -0.1415,  ..., -0.3841, -0.9355,  1.4478],
         [ 0.9647,  1.2974, -1.6207,  ...,  1.1463,  1.5797,  0.3969]],

        [[-0.7713,  0.6572,  0.1663,  ..., -0.8044,  0.0542,  0.7426],
         [ 0.8046,  0.5047,  1.2922,  ...,  1.4648,  0.4097,  0.3205],
         [ 0.0795, -1.7636,  0.5750,  ...,  2.1823,  1.8231, -0.3635],
         [ 0.4267, -0.0647,  0.5686,  ..., -0.5209,  1.3065,  0.8473]],

        ...,

        [[-1.6156,  0.9610, -2.6437,  ..., -0.9645,  1.0888,  1.6383],
         [-0.3985, -0.9235, -1.31

In [ ]:
token_embeddings=token_embedding_layer(inputs)
token_embeddings.shape

torch.Size([8, 4, 256])

In [ ]:
context_length=max_length
pos_embedding_layer=torch.nn.Embedding(context_length,output_dim)

In [ ]:
torch.arange(max_length)

tensor([0, 1, 2, 3])